# 3. 제어 파라미터

**목표** — 같은 프롬프트라도 파라미터를 바꾸면 결과가 어떻게 달라지는지 직접 관찰하고, 용도별로 어떤 값을 써야 하는지 판단할 수 있게 된다.

**소요 시간** 약 90분

| 파라미터 | 무엇을 제어하나 |
| --- | --- |
| `temperature` | 무작위성 — 일관적인가, 창의적인가 |
| `top_p` | 후보 토큰의 범위 (누적 확률) |
| `top_k` | 후보 토큰의 개수 |
| `max_output_tokens` | 출력 길이 상한 → **비용 통제** |
| `stop_sequences` | 특정 문자열이 나오면 중단 |
| `seed` | 재현성 |

> 왜 이런 파라미터가 필요한지는 `[배포용] 1_LLM API 동작 원리와 토큰·과금.md` 2절(샘플링)을 참고한다.
> **반복 호출이 많아서 주로 OpenAI(강사 키)를 쓴다.** Gemini 무료 한도로는 금방 `429`가 나기 때문이다.

## 0. 준비

In [2]:
import os

import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types
from openai import OpenAI

load_dotenv(override=True)

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
oa = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

MODEL = "gemini-3.1-flash-lite"
OA_MODEL = "gpt-4o-mini"


def ask(prompt, **params):
    """OpenAI 호출 헬퍼 — 파라미터만 바꿔가며 실험하기 위한 함수."""
    return oa.responses.create(model=OA_MODEL, input=prompt, **params)


print("준비 완료")

준비 완료


## 1. `temperature` — 무작위성

모델은 매 순간 "다음 토큰의 확률 분포"를 만들고 거기서 하나를 뽑는다.
`temperature`는 **그 분포를 얼마나 뾰족하게/평평하게 만들지**를 정한다.

- **낮음 (0에 가까움)** → 1등 토큰에 쏠림 → 거의 항상 같은 답, 안정적
- **높음** → 하위 후보도 잘 뽑힘 → 다양하고 창의적, 대신 헛소리 위험

같은 질문을 각 설정에서 3번씩 던져 **답이 얼마나 흔들리는지** 본다.

**참고 코드** — 아래를 보고 **다음 셀에 직접 입력**한 뒤 실행한다.

```python
prompt = "동물 이름을 아무거나 하나만 말해줘. 이름만 답해."

for temp in (0.0, 1.0, 1.8):
    answers = [ask(prompt, temperature=temp).output_text.strip() for _ in range(3)]
    print(f"temperature={temp:<4} → {answers}")
```

In [ ]:
# TODO: 위 참고 코드를 직접 입력하고 실행한다
#       확인: temperature 가 높을수록 답이 흩어지는지 본다

In [3]:
prompt = "동물 이름을 아무거나 하나만 말해줘. 이름만 답해."

for temp in (0.0, 1.0, 1.8):
    answers = [ask(prompt, temperature=temp).output_text.strip() for _ in range(3)]
    print(f"temperature={temp:<4} → {answers}")

temperature=0.0  → ['코끼리', '코끼리', '코끼리']
temperature=1.0  → ['코끼리', '코알라', '사자']
temperature=1.8  → ['고양이.', '고양이', '여우']


### 관찰

- `0.0`에서는 3번 모두(혹은 대부분) 같은 답이 나온다 — **하지만 완전한 고정은 아니다.** 내부 연산 순서 등의 이유로 미세하게 달라질 수 있다.
- 값이 커질수록 답이 흩어진다.

> **주의: 허용 범위는 제공자마다 다르다.** OpenAI는 보통 `0.0~2.0`, Gemini도 유사하지만 모델별로 다를 수 있다.
> 범위를 벗어나면 에러가 나므로, 새 모델을 쓸 때는 공식 문서에서 확인한다.

## 2. `top_p` — 후보의 범위 자르기

`top_p`는 확률이 높은 순서로 후보를 더해가다가 **누적 확률이 `p`에 도달하면 거기서 자르는** 방식이다. (nucleus sampling)

```
후보:  서울 82% | 부산 4% | 인천 2% | 광주 1% | ...
top_p=0.8  →  "서울"만 후보로 남음        (거의 결정적)
top_p=1.0  →  모든 후보가 살아있음        (다양)
```

**참고 코드** — 아래를 보고 **다음 셀에 직접 입력**한 뒤 실행한다.

```python
prompt = "새로운 카페 이름을 하나만 지어줘. 이름만 답해."

for p in (0.1, 0.5, 1.0):
    answers = [ask(prompt, top_p=p).output_text.strip() for _ in range(3)]
    print(f"top_p={p:<4} → {answers}")
```

In [ ]:
# TODO: 위 참고 코드를 직접 입력하고 실행한다
#       확인: top_p 가 낮을수록 답이 몰리는지 본다

In [7]:
prompt = "새로운 카페 이름을 하나만 지어줘. 이름만 답해."

for p in (0.1, 0.5, 1.0):
    answers = [ask(prompt, top_p=p).output_text.strip() for _ in range(3)]
    print(f"top_p={p:<4} → {answers}")

top_p=0.1  → ['모닝블렌드', '모닝블렌드', '모닝블렌드']
top_p=0.5  → ['모닝글로리', '모닝블렌드', '모닝드림 카페']
top_p=1.0  → ['커피나무', '모닝커피스무디', '모닝루프']


### `top_k` — 개수로 자르기 (Gemini)

`top_p`가 누적 확률로 자른다면, `top_k`는 **상위 몇 개**로 자른다. `top_k=1`이면 항상 1등만 고른다(그리디).

OpenAI Responses API에는 없고, Gemini는 지원한다.

**참고 코드** — 아래를 보고 **다음 셀에 직접 입력**한 뒤 실행한다.

```python
prompt = "새로운 카페 이름을 하나만 지어줘. 이름만 답해."

for k in (1, 40):
    answers = []
    for _ in range(2):
        r = client.models.generate_content(
            model=MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(top_k=k),
        )
        answers.append(r.text.strip())
    print(f"top_k={k:<3} → {answers}")
```

In [ ]:
# TODO: 위 참고 코드를 직접 입력하고 실행한다
#       확인: top_k=1 이면 항상 같은 답이 나오는지 본다

In [6]:
prompt = "새로운 카페 이름을 하나만 지어줘. 이름만 답해."

for k in (1, 40):
    answers = []
    for _ in range(2):
        r = client.models.generate_content(
            model=MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(top_k=k),
        )
        answers.append(r.text.strip())
    print(f"top_k={k:<3} → {answers}")

top_k=1   → ['오롯이', '오후의 기록']
top_k=40  → ['오브제트', '온도']


### 실무 원칙: `temperature`와 `top_p`를 동시에 만지지 않는다

둘 다 "무작위성을 줄인다"는 같은 목적을 다른 방식으로 수행한다. 동시에 조절하면 **어느 쪽이 결과에 영향을 준 건지 알 수 없어져서** 튜닝이 불가능해진다.

> **하나만 고정하고 다른 하나는 기본값으로 둔다.** 보통 `temperature`를 쓴다.

## 3. `max_output_tokens` — 길이 제한과 비용 통제

출력 토큰의 상한이다. **노트북 02에서 본 것처럼 출력 토큰이 더 비싸므로, 이 값이 비용을 직접 통제한다.**

주의할 점: 이건 "짧게 요약해줘"가 아니라 **"여기서 강제로 끊어라"** 이다. 문장 중간에 잘린다.

**참고 코드** — 아래를 보고 **다음 셀에 직접 입력**한 뒤 실행한다.

```python
prompt = "인공지능의 역사를 설명해줘."

for limit in (20, 100):
    r = ask(prompt, max_output_tokens=limit)
    print(f"--- max_output_tokens={limit} ---")
    print("상태      :", r.status)
    print("중단 사유 :", r.incomplete_details.reason if r.incomplete_details else "없음(정상 종료)")
    print("출력 토큰 :", r.usage.output_tokens)
    print("텍스트    :", repr(r.output_text[:120]))
    print()
```

In [ ]:
# TODO: 위 참고 코드를 직접 입력하고 실행한다
#       확인: status 가 incomplete 이고 잘렸는지 본다

In [9]:
prompt = "인공지능의 역사를 설명해줘."

for limit in (20, 100):
    r = ask(prompt, max_output_tokens=limit)
    print(f"--- max_output_tokens={limit} ---")
    print("상태      :", r.status)
    print("중단 사유 :", r.incomplete_details.reason if r.incomplete_details else "없음(정상 종료)")
    print("출력 토큰 :", r.usage.output_tokens)
    print("텍스트    :", repr(r.output_text[:120]))
    print()

--- max_output_tokens=20 ---
상태      : incomplete
중단 사유 : max_output_tokens
출력 토큰 : 20
텍스트    : '인공지능(AI)의 역사는 여러 단계와 중요한 사건을 포함하여 발전해왔습니다'

--- max_output_tokens=100 ---
상태      : incomplete
중단 사유 : max_output_tokens
출력 토큰 : 100
텍스트    : '인공지능(AI)의 역사는 여러 가지 중요한 발전과 이론이 얽혀 있는 복잡한 이야기입니다. 아래에 주요 사건과 시기를 정리해 보았습니다.\n\n### 1. 초기 아이디어 (1940년대)\n- **앨런 튜링**: 1950년,'



### 여기서 배울 것

- `status`가 `incomplete`이고 `incomplete_details.reason`이 `max_output_tokens`면 **잘린 것이다.**
- 잘린 응답도 **생성된 토큰만큼 과금된다.** 버려질 텍스트에 돈을 낸 셈이다.
- 그래서 실무에서는 **`max_output_tokens`로 상한을 걸되, 프롬프트에서도 "3문장 이내로" 같이 길이를 지시**해서 자연스럽게 끝나게 만든다. 상한은 사고 방지용 안전장치다.

> Gemini에서는 `config=GenerateContentConfig(max_output_tokens=...)`로 지정하고, 잘렸는지는 `finish_reason == 'MAX_TOKENS'`로 확인한다.

## 4. `stop_sequences` — 특정 문자열에서 멈추기

지정한 문자열이 나오면 즉시 생성을 중단한다. **출력 형식을 강제할 때** 유용하다.
(Gemini로 실습한다 — `GenerateContentConfig`에 있다)

**참고 코드** — 아래를 보고 **다음 셀에 직접 입력**한 뒤 실행한다.

```python
prompt = "1부터 5까지 한 줄에 하나씩 숫자만 출력해줘."

for stops in (None, ["3"]):
    r = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(stop_sequences=stops),
    )
    print(f"--- stop_sequences={stops} ---")
    print(r.text)
    print("종료 이유:", r.candidates[0].finish_reason)
    print()
```

In [ ]:
# TODO: 위 참고 코드를 직접 입력하고 실행한다
#       확인: '3' 에서 멈추는지, 중단 문자열이 출력에 포함되는지 본다

In [11]:
for stops in (None, ["3"]):
    r = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(stop_sequences=stops),
    )
    print(f"--- stop_sequences={stops} ---")
    print(r.text)
    print("종료 이유:", r.candidates[0].finish_reason)
    print()

--- stop_sequences=None ---
인공지능(AI)의 역사는 단순히 기술적인 발전을 넘어, 인간이 '지능이란 무엇인가'를 탐구해 온 철학적이고 과학적인 여정입니다. 크게 6단계로 나누어 설명해 드릴게요.

---

### 1. 태동기: 인공지능의 개념 정립 (1940년대 ~ 1950년대)
*   **기초 마련:** 1943년 워런 맥컬록과 월터 피츠가 '뉴런의 논리적 계산'을 발표하며 인공 신경망의 이론적 기초를 닦았습니다.
*   **앨런 튜링:** 1950년 튜링 테스트를 제안하며 "기계가 생각할 수 있는가?"라는 질문을 던졌습니다.
*   **다트머스 회의 (1956):** 존 매카시, 마빈 민스키 등이 모여 **'인공지능(Artificial Intelligence)'**이라는 용어를 처음 사용하며 학문 분야로서의 AI가 공식 출범했습니다.

### 2. 초기 낙관론과 첫 번째 암흑기 (1960년대 ~ 1970년대)
*   **낙관론:** 당시 연구자들은 "10년 안에 인간 수준의 AI가 나올 것"이라며 큰 기대를 했습니다. 체스 게임이나 간단한 수학 문제 풀이 등에서 성과를 보였습니다.
*   **한계 봉착:** 당시의 컴퓨팅 파워로는 복잡한 문제를 해결하기 어려웠고, 실생활 데이터 처리 능력이 부족함이 드러났습니다. 
*   **첫 번째 겨울:** 연구 성과가 기대에 미치지 못하자 정부와 기관의 지원이 끊기며 'AI의 첫 번째 겨울'이 찾아왔습니다.

### 3. 전문가 시스템의 등장 (1980년대)
*   **전문가 시스템:** 특정 분야(의학 진단, 지질 탐사 등)의 지식을 규칙(Rule) 형태로 프로그래밍한 '전문가 시스템'이 상업적으로 성공하며 AI가 다시 주목받았습니다.
*   **한계:** 하지만 이 시스템은 새로운 상황에 유연하게 대처하지 못했고, 유지 보수가 너무 어렵다는 단점이 드러나며 다시 쇠퇴기를 맞이했습니다.

### 4. 머신러닝의 부상 (1990년대 ~ 2000년대 초)
*   **데이터 중심의 전환:** 규칙을

`"3"`을 만나는 순간 멈춘다. **중단 문자열 자체는 출력에 포함되지 않는다.**

## 5. `seed` — 재현성

같은 입력에 같은 결과를 받고 싶을 때 쓴다. 테스트 코드를 짤 때 특히 중요하다.

> 지원 여부와 보장 수준은 **모델과 API마다 다르다.** "같은 seed면 항상 동일"이 아니라 "동일할 가능성이 높다" 정도로 이해하고, 아래처럼 **직접 실행해서 확인하는 습관**을 들인다.

**참고 코드** — 아래를 보고 **다음 셀에 직접 입력**한 뒤 실행한다.

```python
# 답이 여러 갈래로 갈리는 질문이어야 seed 효과가 드러난다.
# ("색깔 하나만 말해줘" 같은 질문은 모델이 한 답에 강하게 쏠려서 차이가 안 보인다)
prompt = "새로 여는 카페 이름을 하나만 지어줘. 이름만."

for label, cfg in [
    ("seed 없음", types.GenerateContentConfig(temperature=2.0)),
    ("seed=42  ", types.GenerateContentConfig(temperature=2.0, seed=42)),
]:
    answers = [
        client.models.generate_content(model=MODEL, contents=prompt, config=cfg).text.strip()
        for _ in range(3)
    ]
    print(f"{label} → {answers}")
```

In [ ]:
# TODO: 위 참고 코드를 직접 입력하고 실행한다
#       확인: seed 를 준 쪽이 더 일관적인지 본다

In [12]:
prompt = "새로 여는 카페 이름을 하나만 지어줘. 이름만."

for label, cfg in [
    ("seed 없음", types.GenerateContentConfig(temperature=2.0)),
    ("seed=42  ", types.GenerateContentConfig(temperature=2.0, seed=42)),
]:
    answers = [
        client.models.generate_content(model=MODEL, contents=prompt, config=cfg).text.strip()
        for _ in range(3)
    ]
    print(f"{label} → {answers}")

seed 없음 → ['무드온', '오후의 서재', '온점']
seed=42   → ['오롯', '오롯', '오롯']


## 6. Gemini로 같은 파라미터 다루기

여기까지 `temperature` / `top_p` / `max_output_tokens`는 **OpenAI로** 실습했다. 반복 호출이 많아 무료 한도에 걸리기 때문이다.

그런데 **17일차 챗봇은 Gemini 전용이다.** 개념은 같아도 **이름과 넣는 자리가 다르므로** 한 번은 직접 넣어봐야 한다.

| 개념 | OpenAI Responses | Gemini |
| --- | --- | --- |
| 넣는 위치 | 호출 인자에 직접 | **`config=GenerateContentConfig(...)`** 로 감싼다 |
| 무작위성 | `temperature=` | `temperature=` (같다) |
| 후보 범위 | `top_p=` | `top_p=` (같다) |
| 후보 개수 | 없음 | `top_k=` |
| 출력 길이 | `max_output_tokens=` | `max_output_tokens=` (같다) |
| 지침 | `instructions=` | **`system_instruction=`** |
| 중단 문자열 | `stop=` | **`stop_sequences=`** (리스트) |
| 잘렸는지 확인 | `incomplete_details.reason` | **`candidates[0].finish_reason`** |
| 사용량 | `usage.input_tokens` | **`usage_metadata.prompt_token_count`** |

**가장 많이 틀리는 곳은 첫 줄이다.** Gemini는 파라미터를 호출 인자에 바로 못 넣고 `config=`로 감싸야 한다.

> **주의: 이 절은 Gemini 무료 한도를 소모한다.** 아래 두 셀에서 5회를 호출한다. 앞 절들(`top_k`·`stop_sequences`·`seed`)에서 이미 12회를 썼으니, `429`가 나면 1분 기다렸다 다시 실행한다.

**참고 코드** — 아래를 보고 **다음 셀에 직접 입력**한 뒤 실행한다.

```python
# temperature — 같은 질문을 각 설정에서 2번씩 (총 4회 호출)
prompt = "새로 여는 빵집 이름을 하나만 지어줘. 이름만."

for temp in (0.0, 1.8):
    answers = []
    for _ in range(2):
        r = client.models.generate_content(
            model=MODEL,
            contents=prompt,
            # 여기가 핵심 — OpenAI처럼 temperature=를 바로 못 넣고 config로 감싼다
            config=types.GenerateContentConfig(temperature=temp),
        )
        answers.append(r.text.strip())
    print(f"temperature={temp:<4} -> {answers}")
```

In [ ]:
# TODO: 위 참고 코드를 직접 입력하고 실행한다
#       확인: config= 로 감싸야 한다는 점에 주의한다

**참고 코드** — 아래를 보고 **다음 셀에 직접 입력**한 뒤 실행한다.

```python
# max_output_tokens — 강제로 잘라보고 finish_reason으로 확인 (1회 호출)
r = client.models.generate_content(
    model=MODEL,
    contents="인공지능의 역사를 설명해줘.",
    config=types.GenerateContentConfig(max_output_tokens=20),
)

print("종료 이유 :", r.candidates[0].finish_reason)   # MAX_TOKENS 면 잘린 것이다
print("출력 토큰 :", r.usage_metadata.candidates_token_count)
print("텍스트    :", repr(r.text))
```

In [ ]:
# TODO: 위 참고 코드를 직접 입력하고 실행한다
#       확인: finish_reason 이 MAX_TOKENS 인지 본다

### 관찰

- `temperature=0.0`은 두 번이 (거의) 같고, `1.8`은 흩어진다 — **OpenAI에서 본 것과 같은 현상**이다. 모델이 달라도 개념은 그대로 통한다.
- `finish_reason`이 `MAX_TOKENS`면 잘린 것이다. OpenAI의 `incomplete_details.reason == "max_output_tokens"`와 같은 뜻이다.
- 세게 자르면 `r.text`가 **빈 문자열**로 온다 (`max_output_tokens=1`로 확인, 2026-08-16 기준). **17일차와 노트북 05에서 `r.text or ""`로 받는 이유가 이것이다** — 값이 비어도 코드가 죽지 않게 한다.

> **여기서 얻을 것** — "SDK를 갈아끼워도 개념은 그대로다. 다만 **이름과 넣는 자리를 확인해야 한다**." 새 제공자를 쓸 때마다 공식 문서에서 이 대조표를 만드는 습관을 들인다.

## 7. 용도별 권장 조합

지금까지 본 것을 실무 기준으로 정리하면 이렇다. **정답이 아니라 출발점**이고, 실제 값은 서비스마다 튜닝한다.

| 용도 | temperature | max_output_tokens | 이유 |
| --- | --- | --- | --- |
| 데이터 추출 / 분류 / JSON 생성 | **0.0 ~ 0.2** | 짧게 | 매번 같은 형식이어야 파싱이 안전하다 |
| 요약 / 번역 | 0.2 ~ 0.5 | 중간 | 원문에 충실하되 문장은 자연스럽게 |
| 고객 상담 챗봇 | 0.5 ~ 0.8 | 중간 | 딱딱하지 않되 헛소리는 곤란 |
| 카피라이팅 / 브레인스토밍 | **0.9 ~ 1.5** | 넉넉히 | 다양한 안이 나와야 한다 |

같은 작업을 두 설정으로 돌려 차이를 확인해본다.

**참고 코드** — 아래를 보고 **다음 셀에 직접 입력**한 뒤 실행한다.

```python
task = """다음 문장에서 제품명과 불만사항을 뽑아 JSON으로만 답해줘.
형식: {"제품명": "...", "불만사항": "..."}

문장: 지난주에 산 무선이어폰 X200이 왼쪽만 소리가 안 나요."""

for label, temp in [("추출 작업에 맞는 설정 (0.0)", 0.0), ("추출 작업에 안 맞는 설정 (1.8)", 1.8)]:
    print(f"--- {label} ---")
    for _ in range(3):
        print(" ", ask(task, temperature=temp).output_text.strip().replace(chr(10), " "))
    print()
```

In [ ]:
# TODO: 위 참고 코드를 직접 입력하고 실행한다
#       확인: temperature 0.0 쪽 JSON 형식이 안 흔들리는지 본다

`temperature=0.0`은 3번 모두 같은 형식으로 나오지만, 높은 값에서는 형식이 흔들린다.
**형식이 흔들린다는 건 `json.loads()`가 실패한다는 뜻**이고, 서비스에서는 그대로 장애가 된다.

## 8. 연습문제

### 연습 3-1. 용도에 맞는 파라미터 고르기

아래 두 작업 각각에 **적절한 파라미터를 직접 정해서** 호출하고, 왜 그 값을 골랐는지 주석으로 적는다.

- 작업 A: 고객 리뷰에서 별점(1~5)만 뽑아내기
- 작업 B: 신제품 광고 문구 5개 만들기

In [ ]:
review = "배송은 빨랐는데 포장이 좀 아쉬웠어요. 그래도 제품 자체는 만족합니다."

# TODO 작업 A: 별점만 숫자로 뽑아내기 — temperature 값을 정하고 이유를 주석으로 적는다
# 이유:
# a = ask(...)

# TODO 작업 B: 광고 문구 5개 — temperature 값을 정하고 이유를 주석으로 적는다
# 이유:
# b = ask(...)

### 연습 3-2. 비용 절감 실험

같은 질문에 `max_output_tokens`를 `1000` / `100`으로 줘서 호출하고,
**노트북 02에서 만든 `cost()` 함수를 다시 정의해** 비용 차이와 절감률을 계산한다.
잘린 응답이 실제로 쓸 만한지도 함께 판단해본다.

In [ ]:
# TODO: 노트북 02에서 확인한 단가를 다시 채운다
PRICE_IN_PER_1M = None
PRICE_OUT_PER_1M = None
USD_KRW = None


def cost(input_tokens, output_tokens):
    usd = (input_tokens / 1_000_000) * PRICE_IN_PER_1M + (output_tokens / 1_000_000) * PRICE_OUT_PER_1M
    return {"usd": usd, "krw": usd * USD_KRW}


# TODO: max_output_tokens 를 1000 / 100 으로 각각 호출하고 비용을 비교한다
# TODO: 절감률(%)을 출력하고, 100 토큰짜리 응답이 실제로 쓸 만한지 판단해 주석으로 적는다

## 정리

- [ ] `temperature`가 높으면 왜 결과가 흔들리는지 설명할 수 있다
- [ ] `temperature`와 `top_p`를 동시에 조절하면 안 되는 이유를 안다
- [ ] `max_output_tokens`로 잘린 응답을 직접 만들어보고 `finish_reason`으로 확인했다
- [ ] 잘린 응답도 과금된다는 것을 안다
- [ ] 추출 작업에 낮은 `temperature`를 써야 하는 이유를 설명할 수 있다

**다음** → [04_single_vs_multiturn.ipynb](./04_single_vs_multiturn.ipynb)
마지막으로, 대화를 **이어가는** 방법과 그 비용을 다룬다.